In [1]:
# Imports

import helpers.helper_functions as hf
import mne
import os.path as op
from mne.channels import combine_channels
import pandas as pd
from mne.beamformer import make_lcmv, apply_lcmv_epochs
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import hilbert
import helpers.test_circ_plot as circ_plot
import gc
import helpers.stc_helper as stc_helper
import time
from pycircstat2.hypothesis import rayleigh_test
from mne.stats import permutation_t_test
from mne.stats import fdr_correction
import nibabel as nib
from scipy.stats import ttest_1samp
from statsmodels.stats.multitest import fdrcorrection


ss = hf.settings_dict()

In [2]:
# the bottom n percent of voxels to be masked
alpha = 0.05

# time frame of interest
stim_tmin = 0.1
stim_tmax = 4.0

base_tmin = -0.5
base_tmax = 0.0

n_permutations = 2**ss['n_trials']


In [3]:
for subject_index in ss['subject_idx_list']:
    subject = ss['subject_id_list'][subject_index]
    print("loading dataset for subject: ", subject)

    for event_id in ss['event_id_list']:
        event_name = str(event_id)
        print("loading event: ", event_name)

        # load stc data
        effect_data = []

        for i in range(ss['n_trials']):
            stc_fname = Path(ss['stc_dir']) / subject / event_name / f"{subject}-{i}-vol.stc"
            stc = mne.read_source_estimate(stc_fname)
            base_stc = stc.copy().crop(tmin=base_tmin, tmax=base_tmax)
            stim_stc = stc.copy().crop(tmin=stim_tmin, tmax=stim_tmax)

            effect = np.sqrt(np.mean(np.square(stim_stc.data), axis=1)) - np.sqrt(np.mean(np.square(base_stc.data), axis=1))
            effect_data.append(effect)

        effect_data_stacked = np.stack(effect_data)

        tvals, pvals = ttest_1samp(effect_data_stacked, popmean=0, axis=0)

        # reject, pvals_fdr = fdrcorrection(pvals, alpha=alpha)

        mask = pvals >= alpha

        active_voxel_count = len(mask) - np.sum(mask)
        print("active_voxel_count: ", active_voxel_count)
        print("total voxel count: ", len(mask))

        # Create dataframe
        df = pd.DataFrame({
            "voxels": mask,
        })

        save_dir = Path(ss['stc_dir']) / subject / event_name
        save_dir.mkdir(parents=True, exist_ok=True)

        # Save to CSV
        df.to_csv(save_dir / f"{subject}-event-{event_name}-mask.csv" , index=False)

loading dataset for subject:  0005_3SJ
loading event:  1
active_voxel_count:  269
total voxel count:  1440
loading event:  2
active_voxel_count:  427
total voxel count:  1440
loading event:  3
active_voxel_count:  310
total voxel count:  1440
loading event:  4
active_voxel_count:  250
total voxel count:  1440
loading event:  5
active_voxel_count:  267
total voxel count:  1440
loading event:  6
active_voxel_count:  384
total voxel count:  1440
loading event:  7
active_voxel_count:  230
total voxel count:  1440
loading dataset for subject:  0002_TCZ
loading event:  1
active_voxel_count:  343
total voxel count:  1868
loading event:  2
active_voxel_count:  142
total voxel count:  1868
loading event:  3
active_voxel_count:  394
total voxel count:  1868
loading event:  4
active_voxel_count:  480
total voxel count:  1868
loading event:  5
active_voxel_count:  350
total voxel count:  1868
loading event:  6
active_voxel_count:  352
total voxel count:  1868
loading event:  7
active_voxel_count: 